## Tracking of a EURUSD dataset optimised over a parameter grid, using likelihoods

In [1]:
## Non-optimised params
optimise_over_T_timesteps= 500 #If using binder, change to lower num of steps (200 confimred to work)
filter_over_T_timesteps= 1000
seed = 1 # Random seem for reproducibility
c=10

In [2]:
## Importing data and converting to a track
from datetime import datetime, timedelta
import os
import pandas as pd
from stonesoup.types.detection import Detection
from stonesoup.types.groundtruth import GroundTruthState
from stonesoup.models.base_driver import NoiseCase
from stonesoup.models.driver import AlphaStableNSMDriver 
from stonesoup.models.transition.levy_linear import LevyLangevin, CombinedLinearLevyTransitionModel
from stonesoup.models.measurement.linear import LinearGaussian
import numpy as np

# Define Predictor, Resampler, and Updater
from stonesoup.predictor.particle import MarginalisedParticlePredictor
from stonesoup.resampler.particle import SystematicResampler
from stonesoup.types.track import Track
from stonesoup.updater.particle import MarginalisedParticleUpdater
# Particle Initialization
from scipy.stats import multivariate_normal
from stonesoup.types.numeric import Probability  # Similar to a float type
from stonesoup.types.state import MarginalisedParticleState
from stonesoup.types.array import StateVector, StateVectors
from stonesoup.predictor.kalman import KalmanPredictor
from stonesoup.models.transition.linear import ConstantVelocity, RandomWalk
from stonesoup.types.state import GaussianState
from stonesoup.updater.kalman import KalmanUpdater
from scipy.stats import norm
from scipy.special import logsumexp



In [ ]:
############################################################
# 0.  Imports (minimal – add any others you already have)
############################################################
import os, copy, numpy as np, pandas as pd
from datetime   import timedelta
from functools  import lru_cache
from scipy.stats      import lognorm
from scipy.optimize   import minimize
from scipy.special    import logsumexp

from stonesoup.types.state           import (GaussianState,
                                             MarginalisedParticleState)
from stonesoup.types.array           import StateVectors, CovarianceMatrices
from stonesoup.types.detection       import Detection
from stonesoup.types.track           import Track
from stonesoup.types.numeric         import Probability
from stonesoup.types.hypothesis      import SingleHypothesis

from stonesoup.resampler.particle    import SystematicResampler
from stonesoup.updater.kalman        import KalmanUpdater
from stonesoup.updater.particle      import MarginalisedParticleUpdater
from stonesoup.predictor.kalman      import KalmanPredictor
from stonesoup.predictor.particle    import MarginalisedParticlePredictor
from stonesoup.models.measurement.linear import LinearGaussian
from stonesoup.models.transition.linear      import RandomWalk
from stonesoup.models.transition.levy_linear import LevyRandomWalk
from stonesoup.models.driver         import AlphaStableNSMDriver

############################################################
# 1.  Load tick data  (EUR/CHF example)
############################################################
folder     = "TrackedDatasets"
file_name  = "EURCHF_Ticks_03.02.2025-03.02.2025.csv"
data       = pd.read_csv(os.path.join(folder, file_name))

tick_times = data['Local time']
mid_rates  = data['Mid']
ask_rates  = data['Ask']
bid_rates  = data['Bid']

filter_over_T_timesteps   = min(filter_over_T_timesteps, len(tick_times))
optimise_over_T_timesteps = min(optimise_over_T_timesteps,
                                filter_over_T_timesteps)

mid_rates = mid_rates.iloc[:filter_over_T_timesteps].to_numpy()
fmt       = r"%d.%m.%Y %H:%M:%S.%f GMT%z"
timesteps = pd.to_datetime(tick_times.iloc[:filter_over_T_timesteps], format=fmt)
start_time = timesteps[0]


############################################################
# 2.  Empirical measurement noise (σₑ) and its prior
############################################################
diffs         = np.diff(mid_rates)
residuals     = diffs[np.abs(diffs) <= np.percentile(np.abs(diffs), 95)]
meas_sigma    = np.std(residuals, ddof=1)
meas_prior    = lognorm(s=1.0, scale=meas_sigma)

# def logprior_sigma2(s2):
#     s = np.sqrt(s2)
#     return meas_prior.logpdf(s) - 0.5*np.log(s2)

# variances used for numerical integration over σₑ²
sigma2_grid = np.logspace(np.log10(meas_sigma**2)-2,
                          np.log10(meas_sigma**2)+2,
                          10)

############################################################
# 3.  Build measurement models & detections  (keys = σₑ²)
############################################################
measurement_models = {}
measurements_dict  = {s2: [] for s2 in sigma2_grid}

for s2 in sigma2_grid:
    measurement_models[s2] = LinearGaussian(
        ndim_state=1,           # price only
        mapping=(0,),
        noise_covar=np.array([[s2]])
    )

for t, ts in enumerate(timesteps[:filter_over_T_timesteps]):
    rate = mid_rates[t]
    for s2 in sigma2_grid:
        measurements_dict[s2].append(
            Detection(state_vector=np.array([rate]),
                      timestamp=ts,
                      measurement_model=measurement_models[s2])
        )

############################################################
# 4.  Prior states (shared between models)
############################################################
number_particles = 500
prior_mean       = np.array([mid_rates[0]])
prior_covar      = np.diag([(0.001*mid_rates[0])**2])

states  = np.random.multivariate_normal(prior_mean, prior_covar,
                                        number_particles)
covars  = [prior_covar]*number_particles

lp_prior = Track(MarginalisedParticleState(
    state_vector = StateVectors(states.T),
    covariance   = CovarianceMatrices(covars),
    weight       = np.full(number_particles, 1/number_particles),
    timestamp    = start_time - timedelta(milliseconds=1)
))

gp_prior = Track(GaussianState(state_vector=prior_mean,
                               covar=prior_covar,
                               timestamp=start_time - timedelta(milliseconds=1)))

############################################################
# 5.  Utility for tolerant dictionary lookup
############################################################
def closest_key(dct, value, atol=1e-12):
    keys = np.fromiter(dct.keys(), dtype=float)
    idx  = np.argmin(np.abs(keys - value))
    if np.abs(keys[idx] - value) > atol:
        raise KeyError(value)
    return float(keys[idx])

############################################################
# 6.  Cached builders (transition+updaters)
############################################################
@lru_cache(maxsize=None)
def build_gp_objects(sigw2):
    model   = RandomWalk(noise_diff_coeff=sigw2)
    pred    = KalmanPredictor(model)
    updaters = {s2: KalmanUpdater(measurement_models[s2]) for s2 in sigma2_grid}
    return pred, updaters

@lru_cache(maxsize=None)
def build_lp_objects(sigw2, alpha):
    driver = AlphaStableNSMDriver(mu_W=0, sigma_W2=sigw2,
                                  c=10.0, alpha=alpha,
                                  noise_case=NoiseCase(2))
    model  = LevyRandomWalk(driver=driver, noise_diff_coeff=sigw2)
    pred   = MarginalisedParticlePredictor(model)
    resamp = SystematicResampler()
    updaters = {s2: MarginalisedParticleUpdater(measurement_models[s2],
                                                resamp)
                for s2 in sigma2_grid}
    return pred, updaters

############################################################
# 7.  Log-likelihood (p(y|θ) with σₑ² integrated out)
############################################################
def filter_loglike(params, model='gp'):
    if model == 'gp':
        sigw2, = params
        predictor, updaters = build_gp_objects(sigw2)
        base_track = gp_prior
    else:
        sigw2, alpha = params
        predictor, updaters = build_lp_objects(sigw2, alpha)
        base_track = lp_prior

    loglikes = []

    for s2 in sigma2_grid:
        updater = updaters[s2]
        logl    = 0 # logprior_sigma2(s2)

        trk = [base_track[0]]  # fresh copy each σ₂

        for meas in measurements_dict[s2][:optimise_over_T_timesteps]:
            pred  = predictor.predict(trk[-1], timestamp=meas.timestamp)
            hypo  = SingleHypothesis(pred, meas)
            post  = updater.update(hypo)
            trk.append(post)

            mp      = hypo.measurement_prediction
            
            # 1-D measurement → flatten to (N,) where N = #particles (N=1 for GP)
            means   = np.asarray(mp.state_vector).ravel()           # (N,)
            covattr = 'covariance' if hasattr(mp, 'covariance') else 'covar'
            vars_   = np.asarray(getattr(mp, covattr)).reshape(-1)  # (N,)

            y       = meas.state_vector.item()

            if means.size == 1:          # ---- Kalman / GP case --------------
                mu   = means[0]
                var  = vars_[0]
                logl += -0.5*((y-mu)**2/var + np.log(2*np.pi*var))

            else:                        # ---- Particle (MPF) case ------------
                ll_arr = -0.5*((y - means)**2/vars_ + np.log(2*np.pi*vars_))
                logl += logsumexp(ll_arr) - np.log(means.size)

        loglikes.append(logl)

    return logsumexp(loglikes)

############################################################
# 8.  Objectives for SciPy (negative log-posterior)
############################################################
def obj_gp(x):
    return -filter_loglike((np.exp(x[0]),), model='gp')

def obj_lp(x):
    sigw2 = np.exp(x[0])
    alpha = 0.5 + 1.45/(1+np.exp(-x[1])) # maps ℝ → (0.5,1.9)
    if alpha==1.0:
        alpha+=0.05
    return -filter_loglike((sigw2, alpha), model='lp')

############################################################
# 9.  Nelder–Mead optimisation
############################################################
print("\n=== Gaussian RW optimisation ===")
res_gp = minimize(obj_gp,
                  x0=[np.log(meas_sigma**2)],
                  method='Nelder-Mead',
                  options={'maxiter': 20, 'disp': True})

best_sigma_w2_gp = np.exp(res_gp.x[0])
print("Best σ_w² (GP):", best_sigma_w2_gp)

print("\n=== Lévy RW optimisation ===")
res_lp = minimize(obj_lp,
                  x0=[np.log(meas_sigma**2), 1.9],
                  method='Nelder-Mead',
                  options={'maxiter': 50, 'disp': True})

best_sigma_w2_lp = np.exp(res_lp.x[0])
best_alpha_lp    = 0.5 + 1.4/(1+np.exp(-res_lp.x[1]))
print("Best (σ_w², α) (LP):", best_sigma_w2_lp, best_alpha_lp)

# ------------------------------------------------------------------
# 1.  Extract best parameters from the optimisers
# ------------------------------------------------------------------
sigma_w2_gp_opt = np.exp(res_gp.x[0])                       # GP σ_w²*
sigma_w2_lp_opt = np.exp(res_lp.x[0])                       # LP σ_w²*
alpha_lp_opt    = 0.5 + 1.4 / (1 + np.exp(-res_lp.x[1]))    # LP α*

print("\nOptimal parameters")
print("  GP : σ_w² =", sigma_w2_gp_opt)
print("  LP : σ_w² =", sigma_w2_lp_opt, ", α =", alpha_lp_opt)

# ──────────────────────────────────────────────────────────────
#  ✧  Single-model objects for the *optimal* parameter set  ✧
# ──────────────────────────────────────────────────────────────
# Pick one measurement variance to use when plotting.
# The empirical value (meas_sigma**2) is a sensible default.
best_sigma2_e = meas_sigma**2

# ── 1.  Measurement model & updaters (ONE per model) ──────────
meas_model_opt = LinearGaussian(
        ndim_state = 1,
        mapping    = (0,),
        noise_covar= np.array([[best_sigma2_e]])
)

gp_updater_opt = KalmanUpdater(meas_model_opt)
lp_updater_opt = MarginalisedParticleUpdater(
        measurement_model = meas_model_opt,
        resampler         = SystematicResampler()
)

# ── 2.  Transition models & predictors with the *optimised* θ ─
gp_predictor_opt = KalmanPredictor(
        RandomWalk(noise_diff_coeff = best_sigma_w2_gp)
)

lp_driver_opt = AlphaStableNSMDriver(
        mu_W      = 0.0,
        sigma_W2  = best_sigma_w2_lp,
        c         = 10.0,                 # same c you used before
        alpha     = best_alpha_lp,
        noise_case= NoiseCase(2)
)
lp_predictor_opt = MarginalisedParticlePredictor(
        LevyRandomWalk(driver = lp_driver_opt,
                       noise_diff_coeff = best_sigma_w2_lp)
)

# ── 3.  Convenience dicts the plotting code expects ───────────
gp_updater  = gp_updater_opt          # <- SINGLE object, not a dict
lp_updater  = lp_updater_opt
gp_predictor = gp_predictor_opt
lp_predictor = lp_predictor_opt

# ──────────────────────────────────────────────────────────────
#  ✧  Observed price tracks & Detection list (ONE σₑ²)  ✧
#      – mid-price is used as the “measurement’’ –
# ──────────────────────────────────────────────────────────────
observed_mid = Track()        # for plotting reference
observed_bid = Track()
observed_ask = Track()

measurements_opt = []         # ← this replaces the old measurements_dict

# ensure bid/ask arrays are NumPy (makes indexing easy)
mid_arr = np.asarray(mid_rates)
bid_arr = np.asarray(bid_rates)
ask_arr = np.asarray(ask_rates)

for price_mid, price_bid, price_ask, ts in zip(mid_arr, bid_arr, ask_arr, timesteps):
    # ground-truth style tracks (useful for plotter)
    observed_mid.append(GroundTruthState([[price_mid]], timestamp=ts))
    observed_bid.append(GroundTruthState([[price_bid]], timestamp=ts))
    observed_ask.append(GroundTruthState([[price_ask]], timestamp=ts))

    # ONE Detection object per time-step, using the optimal σₑ² model
    measurements_opt.append(
        Detection(state_vector=np.array([price_mid]),
                  timestamp      = ts,
                  measurement_model = meas_model_opt)
    )

# -----------------------------------------------------------------
#  ✧  Down-stream code changes
# -----------------------------------------------------------------
# • In your filtering loop, iterate over `measurements_opt`
#   instead of `measurements_dict[ … ]`.
# • You no longer need the big `measurement_models` / `measurements_dict`
#   structures – delete their creation.
# • Plotting keeps using `observed_mid / bid / ask` exactly as before.


# ------------------------------------------------------------------
# 2.  Choose a single measurement variance for *plotting*
# ------------------------------------------------------------------
sigma2_meas_plot = meas_sigma**2           ### choose σe² here ###
meas_model_plot  = measurement_models[closest_key(measurement_models,
                                                  sigma2_meas_plot)]

gp_updater_plot  = gp_updater
lp_updater_plot  = lp_updater

# ------------------------------------------------------------------
# 3.  Build optimal predictors
# ------------------------------------------------------------------
gp_predictor_opt, _ = build_gp_objects(sigma_w2_gp_opt)
lp_predictor_opt, _ = build_lp_objects(sigma_w2_lp_opt, alpha_lp_opt)

# fresh copies of the priors
LP_track = Track(copy.deepcopy(lp_prior[0]))
GP_track = Track(copy.deepcopy(gp_prior[0]))

# ------------------------------------------------------------------
# 4.  Single-pass filtering for the *whole* data set
# ------------------------------------------------------------------
for i, meas in enumerate(measurements_opt):
    # ---- Lévy (MPF) update ---------------------------------------
    lp_pred = lp_predictor_opt.predict(LP_track[-1], timestamp=meas.timestamp)
    lp_hypo = SingleHypothesis(lp_pred, meas)
    lp_post = lp_updater_plot.update(lp_hypo)
    LP_track.append(lp_post)

    # ---- Gaussian (Kalman) update --------------------------------
    gp_pred = gp_predictor_opt.predict(GP_track[-1], timestamp=meas.timestamp)
    gp_hypo = SingleHypothesis(gp_pred, meas)
    gp_post = gp_updater_plot.update(gp_hypo)
    GP_track.append(gp_post)

# ------------------------------------------------------------------
# 5.  Plotting
# ------------------------------------------------------------------
from stonesoup.plotter import Plotterly, Dimension
from pathlib import Path

folder_path = r"C:\Users\joesb\OneDrive\Documents\Cambridge\IIB\PROJECT- Implementation of N-G TAs in SS framework\EURUSDplots"
file_path   = Path(folder_path) / "FinalReportRandom.html"
file_path.parent.mkdir(parents=True, exist_ok=True)

plotter = Plotterly(autosize=False, width=1200, height=800,
                    dimension=Dimension.ONE, axis_labels=["Price"])

# observed mid / bid / ask
plotter.plot_ground_truths(observed_mid, [0], truths_label="Mid")
plotter.plot_ground_truths(observed_bid, [0], truths_label="Bid")
plotter.plot_ground_truths(observed_ask, [0], truths_label="Ask")

# filtered tracks
plotter.plot_tracks(LP_track, [0], mode="lines",
                    uncertainty=True, particle=False,
                    track_label="Lévy filtered", line=dict(width=1))
plotter.plot_tracks(GP_track, [0], mode="lines",
                    uncertainty=True, particle=False,
                    track_label="Gaussian filtered", line=dict(width=1))

# ---- aesthetics --------------------------------------------------
plotter.fig.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray",
               title=dict(text="Time", font=dict(size=20))),
    yaxis=dict(showgrid=True, gridcolor="lightgray",
               title=dict(text="Price", font=dict(size=20))),
    legend=dict(font=dict(size=15),
                bordercolor="Black", borderwidth=2)
)

plotter.fig.write_html(str(file_path))   # ⇐ uncomment to save
plotter.fig.show()



=== Gaussian RW optimisation ===
Optimization terminated successfully.
         Current function value: -4754.173931
         Iterations: 16
         Function evaluations: 32
Best σ_w² (GP): 7.366673986242856e-10

=== Lévy RW optimisation ===


KeyboardInterrupt: 